# TP2 - Construire l'auto-attention

On passe d'une moyenne causale a une attention dont les poids sont appris par Q, K et V.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)


## 1. Agregation et moyenne causale

Une ligne de poids indique la contribution des positions visibles et sa somme vaut 1.

In [ ]:
a = torch.tril(torch.ones(3, 3))
a = a / a.sum(1, keepdim=True)
b = torch.randint(0, 10, (3, 2)).float()
print(a @ b)

xbow_loop = torch.zeros_like(x)
for b in range(B):
    for t in range(T):
        xbow_loop[b, t] = x[b, :t + 1].mean(dim=0)

wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow_matrix = wei @ x
assert torch.allclose(xbow_loop, xbow_matrix, atol=1e-6)
print('moyenne matricielle:', xbow_matrix.shape)


## 2. Masque causal et Softmax

Les `-inf` deviennent des probabilites nulles apres Softmax.

In [ ]:
tril = torch.tril(torch.ones(T, T))
masked = torch.zeros(T, T).masked_fill(tril == 0, float('-inf'))
wei_masked = F.softmax(masked, dim=-1)
xbow_softmax = wei_masked @ x
assert torch.allclose(xbow_loop, xbow_softmax, atol=1e-6)
assert torch.allclose(wei_masked.sum(-1), torch.ones(T))
print('poids causaux valides:', wei_masked[0])


## 3. Une tete d'auto-attention apprise

`Q` exprime la demande d'une position, `K` sa compatibilite et `V` l'information effectivement agregée. La division par `sqrt(dh)` stabilise la variance.

In [ ]:
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k, q, v = key(x), query(x), value(x)
scores = q @ k.transpose(-2, -1) * head_size ** -0.5
scores = scores.masked_fill(tril == 0, float('-inf'))
attention = F.softmax(scores, dim=-1)
out = attention @ v
print('q,k,v:', q.shape, k.shape, v.shape)
print('wei:', attention.shape, 'out:', out.shape)
print('somme:', attention[0].sum(-1))
assert torch.allclose(attention.triu(1), torch.zeros_like(attention.triu(1)))
assert torch.allclose(attention.sum(-1), torch.ones(B, T), atol=1e-6)


In [ ]:
raw = (torch.randn(B, T, head_size) @ torch.randn(B, head_size, T))
scaled = raw * head_size ** -0.5
print('variance brute/scaled:', raw.var().item(), scaled.var().item())
z = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print('softmax:', F.softmax(z, -1))
print('softmax sharp:', F.softmax(z * 8, -1))


### Bilan

Les versions boucle, produit matriciel et masque+Softmax donnent la meme moyenne causale. Une ligne de `attention[0]` correspond a un token requete et ne peut regarder que son passe. Les valeurs `V`, plutot que les cles, portent le contenu a melanger.